# PackSense material-selection preprocessing and readiness

This notebook imports the private food and material reference workbooks using a hash-pinned backend bundle. It then checks whether genuine, reviewed scenario/complete-structure suitability labels and a frozen split are attached. Reference rows are **not** recommendation labels. This notebook does not invent food-material pairs, fit a model, or report accuracy when the evidence is absent.


In [ ]:
import hashlib
import importlib.util
import json
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/packsense_preprocessing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKEND_COMMIT = "730154210c9469f0c96b9aea434cefe0662fd880"

if not importlib.util.find_spec("openpyxl"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "openpyxl==3.1.5"],
        check=True,
    )


def find_unique_input(filename):
    matches = list(INPUT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one attached file named {filename!r}; found {len(matches)}"
        )
    return matches[0]


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# Kaggle attaches the reviewed package as a private dataset, avoiding a GitHub clone.
manifest_path = find_unique_input("source-bundle-manifest.json")
source_root = manifest_path.parent
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest.get("backend_commit") != BACKEND_COMMIT:
    raise ValueError("Backend source manifest does not match the pinned commit.")

verified_source_files = 0
package_init_path = None
for relative_path, expected_hash in manifest["files"].items():
    source_path = source_root / relative_path
    if not source_path.is_file() or sha256_file(source_path) != expected_hash:
        raise ValueError(f"Backend source file failed integrity check: {relative_path}")
    if relative_path.endswith("/packsense/__init__.py"):
        package_init_path = source_path
    verified_source_files += 1

if verified_source_files != len(manifest["files"]) or package_init_path is None:
    raise ValueError("Backend package is incomplete.")

package_dir = package_init_path.parent
if package_dir.name != "packsense" or not (package_dir / "masters.py").is_file():
    raise ValueError("Backend package layout is invalid.")

sys.path.insert(0, str(package_dir.parent))

food_path = find_unique_input(
    "PackSense Food Inputs - USDA Handbook Enriched - 5000 Foods.xlsx"
)
material_path = find_unique_input("PackSense_Packaging_Materials_CO2_Enriched.xlsx")

expected_hashes = {
    food_path.name: "76c5f78c6f6a0ef3c1e5bed7baa442ac9275146198be7de22bbdfe5ef80ccd9f",
    material_path.name: "17e2791aacee70a1e30f6b73624c5ac2745e5feed36672a9c980499d77fd2564",
}
actual_hashes = {path.name: sha256_file(path) for path in (food_path, material_path)}
if actual_hashes != expected_hashes:
    raise ValueError(f"Input workbook hash mismatch: {actual_hashes}")

from packsense.masters import load_food_references, load_material_grades

food_audit = load_food_references(food_path, sheet_name="Food input data")
material_audit = load_material_grades(material_path, sheet_name="Packaging materials")
food_report = food_audit.report()
material_report = material_audit.report()

if food_report["rejected_rows"] or material_report["rejected_rows"]:
    raise ValueError(
        "Reference import rejected rows. Inspect the audit reports before continuing."
    )

# Materialize typed reference records in memory; do not export source rows or train a model.
food_rows = [
    {
        **asdict(entry.reference),
        "source_row_number": entry.source_row_number,
        "pH_evidence": entry.pH_evidence,
    }
    for entry in food_audit.entries
]
food_master = pd.DataFrame(food_rows)


def material_row(entry):
    grade = asdict(entry.grade)
    row = {
        key: value
        for key, value in grade.items()
        if key not in {"otr", "co2tr", "wvtr"}
    }
    row["source_row_number"] = entry.source_row_number
    row["food_application_text"] = entry.food_application_text
    row["barrier_basis_text"] = entry.barrier_basis_text
    row["co2_basis_text"] = entry.co2_basis_text
    row["tensile_basis_text"] = entry.tensile_basis_text
    row["seal_test_note"] = entry.seal_test_note
    for prefix in ("otr", "co2tr", "wvtr"):
        observation = getattr(entry.grade, prefix)
        values = asdict(observation) if observation is not None else {}
        for field in (
            "value",
            "unit",
            "test_temperature_c",
            "test_relative_humidity_pct",
            "test_method",
            "basis",
            "source_url",
            "source_note",
        ):
            value = values.get(field)
            row[f"{prefix}_{field}"] = getattr(value, "value", value)
    return row


material_master = pd.DataFrame(
    [material_row(entry) for entry in material_audit.entries]
)

if len(food_master) != 5000 or food_master["food_id"].nunique() != 5000:
    raise ValueError("Food reference row count or identifier uniqueness check failed.")
if len(material_master) != 81 or material_master["material_id"].nunique() != 81:
    raise ValueError("Material reference row count or identifier uniqueness check failed.")

material_co2 = [entry.grade.co2tr for entry in material_audit.entries]
food_summary = {
    "source_file": food_path.name,
    "source_sha256": actual_hashes[food_path.name],
    "rows_total": int(food_report["total_rows"]),
    "rows_accepted": len(food_audit.entries),
    "rows_rejected": int(food_report["rejected_rows"]),
    "unique_food_ids": int(food_master["food_id"].nunique()),
    "rows_with_moisture": int(food_master["moisture_content_pct"].notna().sum()),
    "rows_with_fat": int(food_master["oil_fat_content_pct"].notna().sum()),
    "rows_with_pH": int(food_master["pH"].notna().sum()),
    "pH_evidence_counts": {
        "missing" if pd.isna(key) else str(key): int(count)
        for key, count in food_master["pH_evidence"].value_counts(dropna=False).items()
    },
    "rows_with_respiration": int(food_master["respiration_rate"].notna().sum()),
}
material_summary = {
    "source_file": material_path.name,
    "source_sha256": actual_hashes[material_path.name],
    "rows_total": int(material_report["total_rows"]),
    "rows_accepted": len(material_audit.entries),
    "rows_rejected": int(material_report["rejected_rows"]),
    "unique_material_ids": int(material_master["material_id"].nunique()),
    "rows_with_OTR": sum(entry.grade.otr is not None for entry in material_audit.entries),
    "rows_with_WVTR": sum(entry.grade.wvtr is not None for entry in material_audit.entries),
    "rows_with_CO2TR": sum(value is not None for value in material_co2),
    "CO2_measured_condition_complete": sum(
        value is not None
        and value.basis.value == "measured"
        and value.test_temperature_c is not None
        and value.test_relative_humidity_pct is not None
        for value in material_co2
    ),
    "CO2_estimated": sum(
        value is not None and value.basis.value == "estimated"
        for value in material_co2
    ),
    "CO2_training_label_rows": sum(
        entry.grade.co2_training_label for entry in material_audit.entries
    ),
}

summary = {
    "preprocessing_stage": "reference_master_import_and_typed_normalization",
    "backend_commit": BACKEND_COMMIT,
    "backend_source_files_verified": verified_source_files,
    "food_reference": food_summary,
    "material_reference": material_summary,
    "training_status": "not_run_reference_tables_are_not_material_suitability_labels",
    "normalized_food_dataframe_shape": [int(value) for value in food_master.shape],
    "normalized_material_dataframe_shape": [int(value) for value in material_master.shape],
}
summary_path = OUTPUT_DIR / "reference_import_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


## Material-model preflight

A valid material model requires a reviewed suitability register, a frozen source/food-grouped split, scenario rows, reviewed complete structures, and explicit source/rights approval. The attached reference workbooks alone are not labelled training data. Missing evidence produces `not_ready`; no training is attempted.


In [ ]:
required_backend_modules = (
    "material_suitability.py",
    "material_split.py",
    "material_model.py",
)
required_training_files = (
    "material-suitability-labels.json",
    "material-split-plan.json",
    "material-scenarios.csv",
    "material-structures.json",
    "material-structure-reviews.json",
    "material-training-approval.json",
)
missing_backend_modules = [
    name for name in required_backend_modules
    if not (package_dir / name).is_file()
]
input_matches = {
    name: list(INPUT_DIR.rglob(name)) for name in required_training_files
}
missing_training_files = [
    name for name, matches in input_matches.items() if len(matches) == 0
]
ambiguous_training_files = [
    name for name, matches in input_matches.items() if len(matches) > 1
]
preflight = {
    "stage": "material_suitability_training_preflight",
    "backend_commit": BACKEND_COMMIT,
    "missing_backend_modules": missing_backend_modules,
    "missing_training_files": missing_training_files,
    "ambiguous_training_files": ambiguous_training_files,
    "status": ("not_ready" if missing_backend_modules or missing_training_files
               or ambiguous_training_files else "inputs_present_for_evidence_audit"),
    "model_trained": False,
    "reason": "reviewed_labels_split_structures_and_approval_required",
}
preflight_path = OUTPUT_DIR / "material_training_preflight.json"
preflight_path.write_text(json.dumps(preflight, indent=2), encoding="utf-8")
print(json.dumps(preflight, indent=2))


## Evidence-gated material training

Only a complete, externally reviewed label set may reach the classifier. The frozen source/food-disjoint split is built before fitting any imputer or encoder. A human approval file must bind the exact label and split-plan hashes and attest source interpretation and training rights. This notebook outputs aggregate diagnostics only; it does not publish a trained artifact or a packaging-safety claim.


In [ ]:
if preflight["status"] == "inputs_present_for_evidence_audit":
    from packsense.enrichment import enrich_scenarios
    from packsense.ingestion import audit_scenarios
    from packsense.material_model import train_material_suitability
    from packsense.material_split import build_material_split, parse_material_split_plan
    from packsense.material_suitability import audit_suitability_labels, parse_suitability_register
    from packsense.structure_review import audit_structure_reviews, parse_structure_review_register
    from packsense.structures import audit_structure_catalogue

    inputs = {name: matches[0] for name, matches in input_matches.items()}
    scenarios = audit_scenarios(inputs["material-scenarios.csv"])
    enriched = enrich_scenarios(scenarios, food_audit)
    grades = {entry.grade.material_id: entry.grade for entry in material_audit.entries}
    catalogue = audit_structure_catalogue(
        inputs["material-structures.json"], grades=grades,
        material_master_sha256=material_audit.source_sha256,
    )
    review_register = parse_structure_review_register(
        inputs["material-structure-reviews.json"].read_bytes()
    )
    reviewed = audit_structure_reviews(catalogue, review_register)
    register = parse_suitability_register(
        inputs["material-suitability-labels.json"].read_bytes()
    )
    label_audit = audit_suitability_labels(
        register, enriched, reviewed,
        material_master_sha256=material_audit.source_sha256,
    )
    plan = parse_material_split_plan(
        inputs["material-split-plan.json"].read_bytes()
    )
    split_audit = build_material_split(
        label_audit, plan,
        food_commodity_by_id={entry.reference.food_id: entry.reference.commodity_type
                              for entry in food_audit.entries},
    )
    def unique_json_object(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f"Duplicate training-approval JSON key: {key}")
            result[key] = value
        return result

    approval = json.loads(
        inputs["material-training-approval.json"].read_text(encoding="utf-8"),
        object_pairs_hook=unique_json_object,
    )
    approval_fields = {"schema_version", "suitability_register_sha256",
                       "split_plan_sha256", "source_documents_checked",
                       "label_decisions_checked", "data_use_rights_checked",
                       "approver_id", "approval_scope"}
    if (not isinstance(approval, dict) or set(approval) != approval_fields
            or type(approval["schema_version"]) is not int or approval["schema_version"] != 1
            or approval["suitability_register_sha256"] != register.source_sha256
            or approval["split_plan_sha256"] != plan.plan_sha256
            or approval["approval_scope"] != "exploratory_material_training"
            or not isinstance(approval["approver_id"], str)
            or not approval["approver_id"].strip()):
        raise ValueError("Material training approval is malformed or bound to different sources.")
    approved = all(type(approval[name]) is bool and approval[name]
                   for name in ("source_documents_checked",
                                "label_decisions_checked",
                                "data_use_rights_checked"))
    training = train_material_suitability(
        label_audit, split_audit, enriched, reviewed, grades,
        source_and_rights_review_approved=approved,
    )
    training_path = OUTPUT_DIR / "material_training_result.json"
    training_path.write_text(json.dumps(training.report, indent=2), encoding="utf-8")
    print(json.dumps(training.report, indent=2))
else:
    print("Material training skipped: required reviewed inputs are not attached.")
